# Exploration des prix — agent Data et indicateurs

Notebook de travail d'Achille (Personne 2). On regarde les données, on prototype les indicateurs ici,
puis la version finale va dans `data/data_agent.py` (`compute_features`).

**Avant de commiter : Clear All Outputs.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(".."))   # racine du repo, pour importer data/ et agents/

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Charger les prix

`USE_MOCK = True` utilise de fausses données (marches aléatoires), pratique pour tester sans internet.
Pour les vrais prix, mettre `False` : il faut alors `data/etf_yahoo_map.csv` (lancer `python data/check_etfs_yfinance.py` une fois).

In [ ]:
USE_MOCK = False
START = date(2025, 1, 1)

if USE_MOCK:
    from data.mock import mock_prices
    df = mock_prices(days=400)
else:
    from data.data_agent import fetch_prices
    df = fetch_prices(START, date.today())

df.head()

In [ ]:
# Un ETF par colonne, comme dans Excel
prix = df.pivot(index="date", columns="ticker", values="close_eur")
prix.index = pd.to_datetime(prix.index)
prix.tail()

## 2. Contrôle de la qualité des données

In [ ]:
resume = df.groupby("ticker").agg(
    premiere_date=("date", "min"),
    derniere_date=("date", "max"),
    nb_jours=("date", "size"),
    dernier_prix=("close_eur", "last"),
)
print(f"{df['ticker'].nunique()} ETF, du {df['date'].min()} au {df['date'].max()}")
resume.sort_values("nb_jours").head(10)   # les ETF avec le moins d'historique

In [ ]:
# Plus fortes variations sur un jour : à vérifier à la main (vrai mouvement ou erreur de données ?)
rdt = prix.pct_change(fill_method=None)
extremes = rdt.abs().stack().sort_values(ascending=False).head(10)
extremes.rename("variation_absolue").to_frame()

## 3. Visualiser

In [ ]:
# Prix ramenés à 100 au départ
base100 = prix / prix.bfill().iloc[0] * 100
choix = ["WORLD", "NASDAQ", "GOLD", "EUR GOVT BOND", "BITCOIN"]
base100[[c for c in choix if c in base100]].plot(figsize=(12, 5), title="Prix en base 100")
plt.ylabel("base 100"); plt.show()

In [ ]:
# Rendement et volatilité annualisés par ETF
stats = pd.DataFrame({
    "rendement_annualise": rdt.mean() * 252,
    "volatilite_annualisee": rdt.std() * np.sqrt(252),
})
ax = stats.plot.scatter(x="volatilite_annualisee", y="rendement_annualise", figsize=(10, 7))
for t, r in stats.iterrows():
    ax.annotate(t, (r["volatilite_annualisee"], r["rendement_annualise"]), fontsize=7)
plt.title("Rendement et volatilité par ETF"); plt.show()

In [ ]:
# Corrélations entre ETF : qui bouge ensemble ?
corr = rdt.corr()
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index, fontsize=7)
fig.colorbar(im); plt.title("Corrélation des rendements journaliers"); plt.show()

## 4. Prototypes des indicateurs

La stratégie retenue : **tendance + timing** (voir `claude/QDC_strategie_agent_technique.md` dans le projet).
On prototype ici, sur le tableau `prix` (une colonne par ETF), puis on déplacera le code dans `compute_features`.

Règle : un indicateur à la date *t* n'utilise que des prix jusqu'à *t* inclus.

In [ ]:
ind = {}

# --- Tendance ---
ind["dist_sma200"] = prix / prix.rolling(200).mean() - 1          # écart à la moyenne 200 jours
ind["mom_3m"] = prix.pct_change(63, fill_method=None)              # momentum 3 mois (63 jours de bourse)
ind["mom_12_1m"] = prix.shift(21).pct_change(231, fill_method=None)  # 12 mois, sans le dernier mois

# --- Timing ---
def rsi(p, n):
    variation = p.diff()
    hausse = variation.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    baisse = (-variation.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + hausse / baisse)

ind["rsi_2"] = rsi(prix, 2)
ind["zscore_20d"] = (prix - prix.rolling(20).mean()) / prix.rolling(20).std()

# --- Risque ---
ind["vol_20d"] = rdt.rolling(20).std() * np.sqrt(252)
ind["drawdown_60d"] = prix / prix.rolling(60).max() - 1

# À faire : bêta au MSCI World, rangs entre ETF, régime de marché

# Coup d'oeil sur un ETF
etf = "WORLD" if "WORLD" in prix else prix.columns[0]
pd.DataFrame({k: v[etf] for k, v in ind.items()}).dropna(how="all").tail(10)

In [ ]:
# Le RSI 2 jours et le z-score sur 6 mois pour un ETF
etf = "NASDAQ" if "NASDAQ" in prix else prix.columns[0]
fen = prix.index[-126:]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
prix.loc[fen, etf].plot(ax=axes[0], title=etf)
ind["rsi_2"].loc[fen, etf].plot(ax=axes[1], title="RSI 2 jours"); axes[1].axhline(10, ls="--"); axes[1].axhline(90, ls="--")
ind["zscore_20d"].loc[fen, etf].plot(ax=axes[2], title="Z-score 20 jours"); axes[2].axhline(-2, ls="--"); axes[2].axhline(2, ls="--")
plt.tight_layout(); plt.show()

In [ ]:
# Format attendu par le pipeline (FEATURES) : une ligne par date, ETF et indicateur
features = (pd.concat({k: v.stack() for k, v in ind.items()}, names=["feature", "date", "ticker"])
            .rename("value").reset_index().dropna())
features = features[["date", "ticker", "feature", "value"]]
features.head()